# 🥇 GOLD Notebook 3 of 3
### Rio Mart · 10-minute grocery delivery

**Rule of Gold: answer the question.**
Small, aggregated tables, each **named after the decision it supports**.

The business stakeholder asked three questions so one table per question is built.

| # | Question | Gold table |
|---|----------|------------|
| 1 | Which city is breaking the 10-minute promise? | `gold_city_performance` |
| 2 | Who are my best and worst riders? | `gold_rider_scorecard` |
| 3 | Which categories actually make the money? | `gold_category_sales` |

```
  /Volumes  →  🥉 BRONZE  →  🥈 SILVER  →  🥇 GOLD  →  📊 Dashboard
                                         you are here
```

⚠️ Run `01_bronze_layer` and `02_silver_layer` first.

In [0]:
CATALOG = "workspace"
SCHEMA  = "rio_mart_quick_commerce"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

from pyspark.sql import functions as F

# on_time_pct, used by two Gold tables.
# is_late is NULL for cancelled orders, so avg() quietly ignores them — meaning
# on-time is measured only across orders we actually delivered.
on_time_pct = F.round(
    100 * F.avg(F.when(F.col("is_late") == False, 1.0)
                 .when(F.col("is_late") == True, 0.0)), 2
)

## Gold 1 — Which city is breaking the promise?

In [0]:
gold_city_performance = (
    spark.table("silver_orders").alias("o")
        .join(spark.table("silver_stores").alias("s"), "store_id", "inner")
        .groupBy("order_date", "city")
        .agg(
            F.count("*").alias("total_orders"),
            F.sum(F.when(F.col("order_status") == "DELIVERED", 1).otherwise(0)).alias("delivered_orders"),
            F.sum(F.when(F.col("order_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_orders"),
            F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
            on_time_pct.alias("on_time_pct"),
            F.round(F.sum("net_amount"), 2).alias("revenue"),
        )
)

gold_city_performance.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_city_performance")
display(spark.table("gold_city_performance").orderBy("order_date", "city").limit(10))

order_date,city,total_orders,delivered_orders,cancelled_orders,avg_delivery_minutes,on_time_pct,revenue
2026-07-23,Bengaluru,45,43,2,8.98,72.09,25165.02
2026-07-23,Delhi,38,36,2,11.15,41.67,21150.95
2026-07-23,Hyderabad,37,36,1,7.91,80.56,23274.92
2026-07-23,Mumbai,44,42,2,9.8,50.0,28165.3
2026-07-24,Bengaluru,47,44,3,8.57,75.0,26797.65
2026-07-24,Delhi,41,35,6,10.77,42.86,25404.8
2026-07-24,Hyderabad,35,35,0,7.76,80.0,20961.76
2026-07-24,Mumbai,68,65,3,10.21,50.77,40367.26
2026-07-25,Bengaluru,88,78,10,9.44,61.54,59771.88
2026-07-25,Delhi,41,38,3,11.39,31.58,22675.99


## Gold 2 — Who are my best and worst riders?

In [0]:
gold_rider_scorecard = (
    spark.table("silver_orders").alias("o")
        .filter((F.col("order_status") == "DELIVERED") & F.col("rider_id").isNotNull())
        .join(spark.table("silver_riders").alias("r"), "rider_id", "inner")
        .groupBy("rider_id", "rider_name", "city", "vehicle_type")
        .agg(
            F.count("*").alias("deliveries"),
            F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
            on_time_pct.alias("on_time_pct"),
            F.round(F.sum("net_amount"), 2).alias("revenue_delivered"),
        )
        .filter(F.col("deliveries") >= 20)      # too few deliveries to judge fairly
)

gold_rider_scorecard.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_rider_scorecard")
display(spark.table("gold_rider_scorecard").orderBy(F.desc("on_time_pct")).limit(10))

rider_id,rider_name,city,vehicle_type,deliveries,avg_delivery_minutes,on_time_pct,revenue_delivered
R0047,Kiran Rao,Hyderabad,Bike,40,7.61,90.0,23139.42
R0027,Gopal Gupta,Hyderabad,Bike,54,7.68,88.89,31172.41
R0097,Manoj Joshi,Hyderabad,Bike,54,7.72,88.89,28872.01
R0037,Vikas Yadav,Hyderabad,Bike,46,7.52,86.96,27841.26
R0111,Gopal Iyer,Bengaluru,EV Scooter,39,8.07,84.62,24856.31
R0118,Arjun Shetty,Hyderabad,EV Scooter,44,8.03,84.09,23481.81
R0041,Ravi Patil,Bengaluru,Bike,36,8.11,83.33,21718.92
R0091,Ravi Rao,Bengaluru,Bike,40,8.21,82.5,28645.42
R0028,Anil Singh,Hyderabad,Bike,40,8.08,82.5,24902.45
R0117,Sunil Singh,Hyderabad,Bicycle,45,8.01,82.22,23338.01


## Gold 3 — Which categories make the money?

In [0]:
gold_category_sales = (
    spark.table("silver_order_items")
        .filter(F.col("order_status") == "DELIVERED")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("orders"),
            F.sum("quantity").alias("units_sold"),
            F.round(F.sum("line_amount"), 2).alias("revenue"),
        )
)

gold_category_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_category_sales")
display(spark.table("gold_category_sales").orderBy(F.desc("revenue")))

category,orders,units_sold,revenue
Personal Care,1593,2468,557968.5
Baby Care,1127,1652,505227.1
Staples,1583,2430,335079.0
Beverages,1826,2860,331027.75
Household,1457,2235,292971.2
Pet Care,580,815,277198.0
Frozen,1133,1626,263194.9
Dairy,1624,2492,177500.2
Fruits & Vegetables,2057,3345,168383.8
Meat,309,431,119087.25


## Answer the Business Stakeholder

The pipeline is done. From here it is just SQL.

In [0]:
%sql
-- Q1: Which city is breaking the 10-minute promise?
SELECT
  city,
  sum(total_orders)                   AS orders,
  round(avg(avg_delivery_minutes), 2) AS avg_delivery_minutes,
  round(avg(on_time_pct), 2)          AS on_time_pct
FROM gold_city_performance
GROUP BY city
ORDER BY on_time_pct ASC

city,orders,avg_delivery_minutes,on_time_pct
Delhi,1112,11.47,30.4
Mumbai,1656,10.12,50.98
Bengaluru,1676,8.97,70.96
Hyderabad,1163,8.34,77.01


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q1b: Is it getting better or worse? Daily trend per city.
SELECT order_date, city, on_time_pct
FROM gold_city_performance
ORDER BY order_date

order_date,city,on_time_pct
2026-07-23,Delhi,41.67
2026-07-23,Hyderabad,80.56
2026-07-23,Mumbai,50.0
2026-07-23,Bengaluru,72.09
2026-07-24,Mumbai,50.77
2026-07-24,Bengaluru,75.0
2026-07-24,Hyderabad,80.0
2026-07-24,Delhi,42.86
2026-07-25,Hyderabad,66.67
2026-07-25,Mumbai,58.73


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q2: The 10 riders who need coaching
SELECT rider_name, city, vehicle_type, deliveries, avg_delivery_minutes, on_time_pct
FROM gold_rider_scorecard
ORDER BY on_time_pct ASC
LIMIT 10

rider_name,city,vehicle_type,deliveries,avg_delivery_minutes,on_time_pct
Kiran Sharma,Delhi,Bike,43,12.05,13.95
Farhan Sharma,Delhi,Bike,31,11.64,19.35
Jitendra Singh,Delhi,Bike,44,12.02,20.45
Harish Das,Delhi,Bike,39,11.99,20.51
Rahul Das,Delhi,EV Scooter,44,11.87,22.73
Anil Shetty,Delhi,EV Scooter,45,11.94,24.44
Naveen Gupta,Delhi,Bike,44,11.65,25.0
Ravi Rao,Delhi,Bike,55,11.21,25.45
Gopal Sharma,Delhi,Bike,51,11.67,25.49
Ajay Sharma,Delhi,Bike,42,12.46,26.19


> Every one of the worst riders is in the same city.

In [0]:
%sql
-- Q3: Which categories make the money?
SELECT category, orders, units_sold, revenue
FROM gold_category_sales
ORDER BY revenue DESC

category,orders,units_sold,revenue
Personal Care,1593,2468,557968.5
Baby Care,1127,1652,505227.1
Staples,1583,2430,335079.0
Beverages,1826,2860,331027.75
Household,1457,2235,292971.2
Pet Care,580,815,277198.0
Frozen,1133,1626,263194.9
Dairy,1624,2492,177500.2
Fruits & Vegetables,2057,3345,168383.8
Meat,309,431,119087.25


Databricks visualization. Run in Databricks to view.

> The most-ordered category is nowhere near the top on revenue. **Volume is not value.**

In [0]:
%sql
-- Time travel: read the table exactly as it was at version 0
SELECT count(*) AS rows_at_version_0
FROM silver_orders VERSION AS OF 0

rows_at_version_0
5607


In [0]:
%sql
-- Time travel: read the table exactly as it was at version 0
SELECT count(*) AS rows_at_version_0
FROM bronze_orders VERSION AS OF 0

rows_at_version_0
5762


## ✅ Builted an end-to-end pipeline

| Layer | Tables | What it gave us |
|---|---|---|
| 🥉 Bronze | 5 | a faithful, replayable copy of the source |
| 🥈 Silver | 4 | clean, deduplicated, business-ready facts |
| 🥇 Gold | 3 | answers, not data |

**None of those three answers existed as a column in the raw files**

### -> Where to go next
1. Schedule these notebooks as a **Job** (Workflows → Create Job) so they run every morning
2. Swap the Bronze read for **Auto Loader** so new files are picked up automatically
3. Build a **Dashboard** on the Gold tables and share it with the business
4. Add **data quality expectations** so bad data never reaches Gold